In [ ]:
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import fastai.tabular.all as ft
from fastai.tabular.all import TabularPandas, RegressionBlock, RandomSplitter, FillMissing
from optuna.storages import RDBStorage
from typing import List, Tuple, Optional

In [ ]:
# Configuración centralizada
class Config:
    """Configuración global para rutas y parámetros fijos"""
    DATA_PATH = "../data/"
    DB_NAME = "wec_optimization.db"
    STORAGE_URL = f"sqlite:///{DB_NAME}"
    TARGET_COLUMN = "Total_Power"
    BATCH_SIZE = 128
    INPUT_FEATURES = 98  # 49 boyas * 2 características

In [ ]:
# Carga de datos
def load_data(file_name: str) -> pd.DataFrame:
    """Carga y valida estructura básica del DataFrame
    
    Args:
        file_name (str): Nombre del archivo CSV a cargar
        
    Returns:
        pd.DataFrame: DataFrame validado
    """
    df = pd.read_csv(Config.DATA_PATH + file_name)
    
    # Validación básica de estructura
    if len(df.columns) < Config.INPUT_FEATURES + 1:
        raise ValueError("El dataset no contiene las columnas esperadas")
        
    return df

In [ ]:
# Conjunto de datos principal
wec_49 = load_data("wec_49.csv")

In [ ]:
def rmse_loss(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """Calcula la pérdida RMSE
    
    Args:
        y_pred: Tensor de predicciones
        y_true: Tensor de valores reales
        
    Returns:
        Tensor con valor de pérdida
    """
    return torch.sqrt(F.mse_loss(y_pred, y_true))

In [ ]:
def baseline(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    """Línea base como media de los valores reales
    
    Args:
        y_pred: Tensor de predicciones (no usado)
        y_true: Tensor de valores reales
        
    Returns:
        Tensor con valor de la métrica
    """
    return torch.mean(y_true)

In [ ]:
class PermuteBuoys(nn.Module):
    """Módulo para permutar características de boyas durante el entrenamiento
    
    Attributes:
        input_features (int): Número total de características de entrada
    """
    def __init__(self, input_features: int = Config.INPUT_FEATURES):
        super().__init__()
        if input_features % 2 != 0:
            raise ValueError("El número de características debe ser par")
        self.n_buoys = input_features // 2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Permuta las boyas durante el entrenamiento
        
        Args:
            x: Tensor de entrada con forma (batch_size, features)
            
        Returns:
            Tensor permutado o original según modo de entrenamiento
        """
        if self.training:
            bs = x.size(0)
            try:
                # Usar reshape en lugar de view y asegurar contigüidad
                x = x.reshape(bs, self.n_buoys, 2).contiguous()
                perm = torch.randperm(self.n_buoys, device=x.device)
                x = x[:, perm, :]
                return x.reshape(bs, -1)
            except RuntimeError as e:
                raise ValueError(
                    f"Error en permutación: {str(e)}\n"
                    f"Forma del tensor: {x.shape}\n"
                    f"Características esperadas: {self.n_buoys * 2}"
                )
        return x

In [ ]:
class ResBlock(nn.Module):
    """Bloque residual con normalización y dropout opcional
    
    Args:
        n_features: Número de características de entrada/salida
        dropout: Probabilidad de dropout (None para desactivar)
    """
    def __init__(self, n_features: int, dropout: Optional[float] = None):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(n_features, n_features),
            nn.BatchNorm1d(n_features),
            nn.ReLU(),
            nn.Linear(n_features, n_features)
        )
        self.dropout = nn.Dropout(dropout) if dropout else None
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.block(x)
        if self.dropout:
            residual = self.dropout(residual)
        return self.activation(x + residual)

In [ ]:
class ResModel(nn.Module):
    """Modelo residual con bloques personalizables
    
    Attributes:
        base_model (nn.Module): Modelo base de fastai
        resblocks (nn.Sequential): Secuencia de bloques residuales
        permute (PermuteBuoys): Módulo de permutación de características
    """
    def __init__(
        self,
        base_model: nn.Module,
        n_resblocks: int = 3,
        input_features: int = Config.INPUT_FEATURES,
        dropout: Optional[float] = None
    ):
        super().__init__()
        self.base_model = base_model
        self.resblocks = nn.Sequential(*[
            ResBlock(input_features, dropout) 
            for _ in range(n_resblocks)
        ])
        self.permute = PermuteBuoys(input_features)

    def forward(self, x_cat: torch.Tensor, x_cont: torch.Tensor) -> torch.Tensor:
        """Procesamiento de características continuas y categóricas
        
        Args:
            x_cat: Features categóricos
            x_cont: Features continuos
            
        Returns:
            Salida del modelo
        """
        x_cont = self.permute(x_cont)
        x_cont = self.resblocks(x_cont)
        return self.base_model(x_cat, x_cont)

In [ ]:
def create_dataloaders(
    df: pd.DataFrame,
    cont_names: List[str],
    y_names: List[str],
    batch_size: int = Config.BATCH_SIZE
) -> Tuple[TabularPandas, torch.utils.data.DataLoader]:
    """Crea dataloaders tabulares con validación
    
    Args:
        df: DataFrame de entrada
        cont_names: Lista de nombres de features continuos
        y_names: Lista de nombres de targets
        batch_size: Tamaño del batch
        
    Returns:
        Tupla con TabularPandas y DataLoader
    """
    if len(cont_names) != Config.INPUT_FEATURES:
        raise ValueError(f"Se esperaban {Config.INPUT_FEATURES} features continuos")
        
    procs = [FillMissing]
    splitter = RandomSplitter()
    
    tab_data = TabularPandas(
        df,
        cont_names=cont_names,
        y_names=y_names,
        y_block=RegressionBlock,
        splits=splitter(df),
        procs=procs
    )
    
    return tab_data.dataloaders(bs=batch_size)

In [ ]:
# Configuración de características
cont_names = wec_49.columns.tolist()[:Config.INPUT_FEATURES]
y_names = [Config.TARGET_COLUMN]
dls = create_dataloaders(wec_49, cont_names, y_names)

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Función objetivo para optimización con Optuna
    
    Args:
        trial: Objeto Trial de Optuna
        
    Returns:
        Valor de la métrica a optimizar
    """
    # Espacio de búsqueda parametrizado
    layers = [
        trial.suggest_int(f"n_units_l{i}", 256, 1024)
        for i in range(trial.suggest_int("n_layers", 3, 5))
    ]
    
    hparams = {
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        "n_resblocks": trial.suggest_int("n_resblocks", 3, 5),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3)
    }
    
    # Configuración del modelo
    metrics = [baseline, F.l1_loss, F.smooth_l1_loss, rmse_loss]
    learn = ft.tabular_learner(
        dls,
        layers=layers,
        metrics=metrics,
        opt_func=ft.ranger
    )
    
    try:
        learn.model = ResModel(
            learn.model,
            n_resblocks=hparams["n_resblocks"],
            dropout=hparams["dropout"]
        )
    except Exception as e:
        trial.set_user_attr("error", str(e))
        raise optuna.exceptions.TrialPruned()
    
    # Entrenamiento con manejo de errores
    try:
        learn.fit_one_cycle(10, hparams["lr"])
        return learn.recorder.values[-1][1]
    except RuntimeError as e:
        trial.set_user_attr("error", str(e))
        raise optuna.exceptions.TrialPruned()

In [ ]:
def setup_study() -> optuna.Study:
    """Configura y retorna estudio de Optuna
    
    Returns:
        Objeto Study configurado
    """
    storage = RDBStorage(Config.STORAGE_URL)
    return optuna.create_study(
        study_name="WEC_NN_v1",
        direction="minimize",
        storage=storage,
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=42)
    )

In [ ]:
if __name__ == "__main__":
    study = setup_study()
    study.optimize(
        objective,
        n_trials=300,
        show_progress_bar=True,
        gc_after_trial=True
    )